In [22]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from pandas_datareader import data as web

# Data Setup

In [ ]:
# Read merged data
file_path = "data/intermediate/anes_individual_level.csv"
df = pd.read_csv(file_path, low_memory=False)

In [29]:
# Leave only trust variables
time_series = [    
    "year",
    
    "trust_gov_right",
    "gov_for_all",
    "gov_waste",
    "crooked_officials",
    "trust_gov_index",

    "trust_gov_right_z",
    "gov_for_all_z",
    "gov_waste_z",
    "crooked_officials_z",
    "trust_gov_index_z",

    "trust_gov_right_d",
    "gov_for_all_d",
    "gov_waste_d",
    "crooked_officials_d",
    "trust_gov_index_d",
]

df_time = df[time_series]
df_time = df_time.groupby('year').mean().reset_index()
df_time= df_time[df_time['year'] > 1956]

In [31]:
# Extract and clean FRED data
def get_recession_periods(min_year: int, max_year: int):
    usrec = web.DataReader('USREC', 'fred',
                           start=datetime(min_year, 1, 1),
                           end=datetime(max_year, 12, 31)).reset_index()
    usrec.rename(columns={'DATE': 'date', 'USREC': 'usrec'}, inplace=True)

# contiguous blocks where usrec changes
    usrec['block'] = (usrec['usrec'].diff().fillna(0) != 0).cumsum()

    periods = []
    for (_, val), g in usrec.groupby(['block', 'usrec']):
        if val == 1:
            periods.append((g['date'].min(), g['date'].max()))
    return periods

In [32]:
# Adds bands to graphs
def add_recession_bands(ax, recession_periods, label='Recession'):
    first = True
    for start, end in recession_periods:
        # Convert to year fractions since your x-axis is year-int
        start_x = start.year + (start.month - 1) / 12.0
        end_x   = end.year   + (end.month - 1) / 12.0
        ax.axvspan(start_x, end_x, alpha=0.25, edgecolor='none',
                   label=(label if first else None))
        first = False

# Graph Plotting

In [69]:
# Main plot function
def plot_ts_one(df_time, year_col, var, recession_periods, outdir,
                y_label, title_prefix="", ylim=None):
    os.makedirs(outdir, exist_ok=True)
    
    df_plot = df_time[[year_col, var]].copy()
    df_plot = df_plot.sort_values(year_col)

    fig, ax = plt.subplots(figsize=(12, 6))

    # recession bands
    add_recession_bands(ax, recession_periods)

    # main line + dots (no interpolation; line breaks at NaNs)
    valid = df_plot.dropna(subset=[var])
    ax.plot(valid[year_col], valid[var],
            marker='o', markersize=4,
            linestyle=':', linewidth=1.5,
            label=var)

    ax.set_title(f"{title_prefix}{var}")
    ax.set_xlabel("Year")
    ax.set_ylabel(y_label)

    if ylim is not None:
        ax.set_ylim(*ylim)

   # --- Fixed core window with small buffer for endpoints ---
    core_left, core_right = 1960, 2020

    # small buffer to keep 1958/2024 visible but not too much empty space
    left_buffer  = 3   # shows 1958 if present
    right_buffer = 5   # shows 2024 if present

    ax.set_xlim(core_left - left_buffer, core_right + right_buffer)
    ax.margins(x=0.01)

    # decade ticks (10-year bins) anchored to the core window
    ax.set_xticks(np.arange(core_left, core_right + 1, 10))

    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right')
    fig.tight_layout()

    outpath = os.path.join(outdir, f"{var}_ts.png")
    fig.savefig(outpath, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [58]:
# Define three groups of trust variables
year_col = "year" 

# Variable lists
trust_vars_raw = ["trust_gov_right", "gov_for_all", "gov_waste", "crooked_officials", "trust_gov_index"]
trust_vars_z   = [v + "_z" for v in trust_vars_raw]
trust_vars_d   = [v + "_d" for v in trust_vars_raw]

min_year = int(df_time[year_col].min())
max_year = int(df_time[year_col].max())
recession_periods = get_recession_periods(min_year, max_year) # year ranges

In [59]:
# Set y-axis limits
# Dummy: fixed (0-1)
ylim_dummy = (-0.05, 1.05)

# Z-score: unified across all z variables 
z_min = np.nanmin(df_time[trust_vars_z].to_numpy())
z_max = np.nanmax(df_time[trust_vars_z].to_numpy())
pad = 0.1 * (z_max - z_min) if np.isfinite(z_max - z_min) and (z_max > z_min) else 0.5
ylim_z = (z_min - pad, z_max + pad)

# Raw: fixed within variable 
valid_ranges = {
    "trust_gov_right": (1, 5),
    "gov_for_all": (1, 2),
    "gov_waste": (1, 3),
    "crooked_officials": (1, 3),
    "trust_gov_index": (0, 100)
}

In [70]:
# Plot loop (original)
for v in trust_vars_raw:
    lo, hi = valid_ranges.get(v, (None, None))
    ylim_raw = (lo - 0.05*(hi-lo), hi + 0.05*(hi-lo)) if (lo is not None and hi is not None) else None
    plot_ts_one(df_time, year_col, v, recession_periods,
                outdir="figures/time_series/original",
                y_label="Average (original scale)",
                title_prefix="Time Series (Raw): ",
                ylim=ylim_raw)



In [71]:
# Plot loop (z-score)
for v in trust_vars_z:
    plot_ts_one(df_time, year_col, v, recession_periods,
                outdir="figures/time_series/z_score",
                y_label="Z-score (national mean)",
                title_prefix="Time Series (Z): ",
                ylim=ylim_z)

In [72]:
# Plot loop (dummy)
for v in trust_vars_d:
    plot_ts_one(df_time, year_col, v, recession_periods,
                outdir="figures/time_series/dummy",
                y_label="Share (dummy=1)",
                title_prefix="Time Series (Dummy): ",
                ylim=ylim_dummy)